Explanation:
This step imports the required libraries for data manipulation, visualization, and preprocessing.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler

Explanation:
Load the dataset again because each notebook should be independent and reproducible.

In [3]:
train_df = pd.read_csv(r"C:/Users/daraa/Desktop/Transformer-based vs. Hybrid Deep Learning Architectures/Data/train.csv")
test_df = pd.read_csv(r"C:/Users/daraa/Desktop/Transformer-based vs. Hybrid Deep Learning Architectures/Data/test.csv")

train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

train_df.head()

,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10


Explanation:
Time-series datasets contain hidden information in the date column.
Extracting features like year, month, or weekday helps models understand seasonal patterns.

In [4]:
train_df['year'] = train_df['date'].dt.year
train_df['month'] = train_df['date'].dt.month
train_df['day'] = train_df['date'].dt.day
train_df['day_of_week'] = train_df['date'].dt.weekday
train_df['week_of_year'] = train_df['date'].dt.isocalendar().week
train_df[['date','year','month','day','day_of_week','week_of_year']].head()

,date,year,month,day,day_of_week,week_of_year
0,2013-01-01,2013,1,1,1,1
1,2013-01-02,2013,1,2,2,1
2,2013-01-03,2013,1,3,3,1
3,2013-01-04,2013,1,4,4,1
4,2013-01-05,2013,1,5,5,1


Explanation:
Lag features allow the model to use previous sales values to predict future sales.
Past observations often strongly influence future demand in time-series forecasting.

In [5]:
train_df['lag_7'] = train_df['sales'].shift(7)
train_df['lag_14'] = train_df['sales'].shift(14)
train_df['lag_30'] = train_df['sales'].shift(30)
train_df[['sales','lag_7','lag_14','lag_30']].head(15)

,sales,lag_7,lag_14,lag_30
0,13,NaN,NaN,NaN
1,11,NaN,NaN,NaN
2,14,NaN,NaN,NaN
3,13,NaN,NaN,NaN
4,10,NaN,NaN,NaN
5,12,NaN,NaN,NaN
6,10,NaN,NaN,NaN
7,9,13.0,NaN,NaN
8,12,11.0,NaN,NaN
9,9,14.0,NaN,NaN


Explanation:
Rolling statistics summarize recent trends in the data.

In [6]:
train_df['rolling_mean_7'] = train_df['sales'].rolling(window=7).mean()
train_df['rolling_std_7'] = train_df['sales'].rolling(window=7).std()
train_df[['sales','rolling_mean_7','rolling_std_7']].head(15)

,sales,rolling_mean_7,rolling_std_7
0,13,NaN,NaN
1,11,NaN,NaN
2,14,NaN,NaN
3,13,NaN,NaN
4,10,NaN,NaN
5,12,NaN,NaN
6,10,11.857143,1.573592
7,9,11.285714,1.799471
8,12,11.428571,1.812654
9,9,10.714286,1.603567


In [7]:
train_df = train_df.dropna()

In [8]:
train_df.shape

(912970, 14)

Explanation:
Separate input features and the target variable.

In [9]:
features = [
    'store',
    'item',
    'year',
    'month',
    'day',
    'day_of_week',
    'week_of_year',
    'lag_7',
    'lag_14',
    'lag_30',
    'rolling_mean_7',
    'rolling_std_7'
]

X = train_df[features]
y = train_df['sales']
print(X.shape)
print(y.shape)

(912970, 12)
(912970,)


Explanation:
Deep learning models perform better when features are normalized.

In [11]:
scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)
print(X_scaled[:5])

[[0.         0.         0.         0.         1.         0.5
  0.07692308 0.03463203 0.06926407 0.05627706 0.03195963 0.05364791]
 [0.         0.         0.         0.09090909 0.         0.66666667
  0.07692308 0.06060606 0.03030303 0.04761905 0.0294365  0.0458758 ]
 [0.         0.         0.         0.09090909 0.03333333 0.83333333
  0.07692308 0.05194805 0.07792208 0.06060606 0.03700589 0.09799811]
 [0.         0.         0.         0.09090909 0.06666667 1.
  0.07692308 0.05194805 0.06493506 0.05627706 0.03952902 0.10148978]
 [0.         0.         0.         0.09090909 0.1        0.
  0.09615385 0.04761905 0.03463203 0.04329004 0.04205214 0.10148978]]


Explanation:
Time-series data must be split chronologically rather than randomly.

In [12]:
split = int(len(X_scaled) * 0.8)

X_train = X_scaled[:split]
X_val = X_scaled[split:]

y_train = y[:split]
y_val = y[split:]
print(X_train.shape)
print(X_val.shape)

(730376, 12)
(182594, 12)


Explanation:
Sequence models require input in the form:

In [13]:
def create_sequences(data, target, window_size):

    X_seq = []
    y_seq = []

    for i in range(len(data) - window_size):

        X_seq.append(data[i:i+window_size])
        y_seq.append(target[i+window_size])

    return np.array(X_seq), np.array(y_seq)

window_size = 30

X_seq, y_seq = create_sequences(X_scaled, y.values, window_size)
print(X_seq.shape)
print(y_seq.shape)

(912940, 30, 12)
(912940,)


In [14]:
np.save("X_seq.npy", X_seq)
np.save("y_seq.npy", y_seq)